# [Chapter 9 — Practical Issues in Fitting, §9.7] Pitfall 7: Recognizing When Stochastic Effects Dominate

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 9 — Practical Issues in Fitting
**Considerations developed:** 9, 13
**Estimated runtime:** ≤ 30s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Deterministic models are inappropriate when the infectious population is small. Demonstrates the divergence between deterministic predictions and stochastic realizations near $\mathcal{R}_0 = 1$ or with small initial $I_0$.

## What you should already know
Chapter 8 fitting notebooks.


## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from shared import book_style, BOOK_COLORS, integrate_sir_i
from shared.parameters import baseline_chapter_08
from shared.seeds import set_seed_chapter_08
from shared.verification import assert_within_tolerance
set_seed_chapter_08()
book_style()


In [ ]:
# Stochastic SIR via tau-leap or Gillespie (simple version)
def stochastic_sir(R0, tau_R=10, S0_count=1000, I0_count=2, t_end=80, dt=0.1):
    np.random.seed(None)  # different each run
    c_I_beta_over_N = R0 / tau_R / S0_count  # alpha = c_I * beta / N
    S, I, R = S0_count, I0_count, 0
    t = 0
    times, S_arr, I_arr, R_arr = [t], [S], [I], [R]
    while t < t_end and I > 0:
        rate_inf = c_I_beta_over_N * S * I
        rate_rec = I / tau_R
        rate_total = rate_inf + rate_rec
        if rate_total <= 0:
            break
        # Tau-leap step
        n_inf = np.random.poisson(rate_inf * dt)
        n_rec = np.random.poisson(rate_rec * dt)
        n_inf = min(n_inf, S)
        n_rec = min(n_rec, I)
        S -= n_inf
        I += n_inf - n_rec
        R += n_rec
        t += dt
        times.append(t); S_arr.append(S); I_arr.append(I); R_arr.append(R)
    return np.array(times), np.array(S_arr), np.array(I_arr), np.array(R_arr)

# Run 50 stochastic realizations near R_0 = 1.5 (near threshold)
R0 = 1.5
n_runs = 50
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
n_extinct = 0
for _ in range(n_runs):
    t_s, _, I_s, _ = stochastic_sir(R0=R0, S0_count=1000, I0_count=2)
    ax.plot(t_s, I_s, color=BOOK_COLORS['infectious'], alpha=0.2, lw=0.7)
    if I_s[-1] == 0 and t_s[-1] < 80:
        n_extinct += 1
ax.set_xlabel('Time (days)')
ax.set_ylabel('I (count)')
ax.set_title(f'Stochastic SIR (R_0={R0}, I_0=2): {n_extinct}/{n_runs} extinct')

# Now with larger I0
ax = axes[1]
n_extinct_large = 0
for _ in range(n_runs):
    t_s, _, I_s, _ = stochastic_sir(R0=R0, S0_count=1000, I0_count=20)
    ax.plot(t_s, I_s, color=BOOK_COLORS['infectious'], alpha=0.2, lw=0.7)
    if I_s[-1] == 0 and t_s[-1] < 80:
        n_extinct_large += 1
ax.set_xlabel('Time (days)')
ax.set_ylabel('I (count)')
ax.set_title(f'Stochastic SIR (R_0={R0}, I_0=20): {n_extinct_large}/{n_runs} extinct')

plt.tight_layout()
plt.show()

print(f"\nWith small I_0=2, {n_extinct}/{n_runs} stochastic outbreaks went extinct early.")
print(f"With larger I_0=20, only {n_extinct_large}/{n_runs} did.")
print(f"Deterministic model predicts an outbreak in EVERY case (R_0 > 1).")
print("\nLesson: when I_0 is small or R_0 is near 1, deterministic predictions overstate certainty.")
